# Preprocessing & Feature Engineering

This notebook prepares interaction and product metadata for recommendation
modeling.

## Goals

1. Define the preprocessing and leakage boundaries.
2. Construct robust product text from available metadata.
3. Handle missing metadata without removing cold-start products.
4. Prepare the inputs required by the recommendation approaches.

## Leakage constraint

Interaction-derived features must use only information available in the
training period (<= 2021).

Product metadata can be used for all catalog items because the project's
cold-start setting assumes product content is available even when historical
interaction data is absent.

## Content representation

Product content will combine:

- title
- description
- features
- categories

Missing fields are omitted rather than causing the product to be removed.

## Product text construction

In [3]:
import duckdb

con = duckdb.connect()

METADATA_PATH = "/home/henry-k/Documents/Projects/cold-start-recommender/data/raw/meta_Electronics.jsonl.gz"

sample_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    categories
FROM read_json_auto('{METADATA_PATH}')
WHERE title IS NOT NULL
LIMIT 5
"""

sample_products = con.execute(sample_query).df()
sample_products

,parent_asin,title,description,features,categories
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,[Teleporter V3 The “Teleporter V3” kit sets a ...,[],"[Electronics, Television & Video, Video Glasses]"
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,[HDMI In - HDMI Out],"[UPC: 662774021904, Weight: 0.600 lbs]","[Electronics, Television & Video, Accessories,..."
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,[],[WARNING: Please IDENTIFY MODEL NUMBER on the ...,"[Electronics, Computers & Accessories, Laptop ..."
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,[],[☛NotoCity 22mm band is designed for Vivoactiv...,"[Electronics, Wearable Technology, Clips, Arm ..."
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,[all Genuine High Quality Motorola Made Access...,"[New Droid X Essentials Combo Pack, Exclusive ...","[Electronics, Computers & Accessories, Compute..."


In [6]:
sample_products[["description", "features", "categories"]].applymap(
    lambda x: type(x).__name__
)

/tmp/ipykernel_36486/2486680878.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  sample_products[["description", "features", "categories"]].applymap(


,description,features,categories
0,ndarray,ndarray,ndarray
1,ndarray,ndarray,ndarray
2,ndarray,ndarray,ndarray
3,ndarray,ndarray,ndarray
4,ndarray,ndarray,ndarray


In [7]:
for column in ["description", "features", "categories"]:
    print(f"\n--- {column} ---")
    for value in sample_products[column].head(5):
        print(type(value).__name__, ":", value)


--- description ---
ndarray : ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.']
ndarray : ['HDMI In - HDMI Out']
ndarray : []
ndarray : []
ndarray : ['all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter C

In [8]:
def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, np.ndarray):
        parts = [
            str(item).strip()
            for item in value
            if item is not None and str(item).strip()
        ]
        return " ".join(parts)

    text = str(value).strip()
    return " ".join(text.split())

In [9]:
import numpy as np

In [13]:
def build_product_text(row):
    sections = []

    title = clean_text(row["title"])
    description = clean_text(row["description"])
    features = clean_text(row["features"])
    categories = " > ".join(
        clean_text(category)
        for category in row["categories"]
        if clean_text(category)
    )

    if title:
        sections.append(f"Title: {title}")

    if description:
        sections.append(f"Description: {description}")

    if features:
        sections.append(f"Features: {features}")

    if categories:
        sections.append(f"Categories: {categories}")

    return "\n".join(sections)

In [14]:
sample_products["product_text"] = sample_products.apply(
    build_product_text,
    axis=1
)

for _, row in sample_products[["parent_asin", "product_text"]].iterrows():
    print("=" * 80)
    print(row["parent_asin"])
    print(row["product_text"])

B00MCW7G9M
Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET
Description: Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.
Categories: Electronics > Television & Video > Video Glasses
B00YT6XQSE
Title: Ce-H22B12-S1 4Kx2K Hdmi 4Port
Description: HDMI In - HDMI Out
Features: UPC: 662774021904 Weight: 0.600 lbs
Categories: Electronics > Television & Video > Accessories > Cables > HDMI Cables
B0

## Product text length analysis

Before selecting a truncation strategy, we inspect the length distribution of
the constructed product text.

This helps determine whether most products fit comfortably within the eventual
embedding model's context limit or whether long-product truncation will be a
significant preprocessing decision.

In [15]:
length_query = f"""
SELECT
    parent_asin,
    length(COALESCE(title, '')) AS title_chars,
    length(
        CASE
            WHEN description IS NULL THEN ''
            ELSE array_to_string(description, ' ')
        END
    ) AS description_chars,
    length(
        CASE
            WHEN features IS NULL THEN ''
            ELSE array_to_string(features, ' ')
        END
    ) AS features_chars,
    length(
        CASE
            WHEN categories IS NULL THEN ''
            ELSE array_to_string(categories, ' > ')
        END
    ) AS categories_chars
FROM read_json_auto('{METADATA_PATH}')
"""

text_lengths = con.execute(length_query).df()
text_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,title_chars,description_chars,features_chars,categories_chars
count,1.610012e+06,1.610012e+06,1.610012e+06,1.610012e+06
mean,1.232270e+02,4.557688e+02,4.658179e+02,7.631475e+01
std,5.701154e+01,9.699949e+02,5.174651e+02,3.191127e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.220000e+02,9.700000e+01,2.940000e+02,7.900000e+01
90%,1.940000e+02,1.320000e+03,1.227000e+03,1.120000e+02
95%,1.980000e+02,1.698000e+03,1.518000e+03,1.270000e+02
99%,2.100000e+02,3.687000e+03,2.084000e+03,1.500000e+02
max,2.000000e+03,2.030380e+05,9.770000e+03,1.570000e+02


## Semantic embedding model

We use a small pretrained Sentence Transformer model to convert product text
into dense semantic vectors.

A small encoder is chosen because the catalog contains approximately 1.6
million products. A smaller embedding model reduces local inference time,
GPU memory requirements, and embedding storage while keeping the content
representation sufficiently expressive for the first experiment.

The model choice can be revisited if evaluation shows that semantic
representation quality is a limiting factor.

In [16]:
import torch
import sentence_transformers

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Sentence Transformers:", sentence_transformers.__version__)

PyTorch: 2.14.0+cu130
CUDA available: True
Sentence Transformers: 6.1.0


## Load semantic embedding model

We use `all-MiniLM-L6-v2` as the initial semantic encoder.

The model is intentionally small because embeddings will eventually be generated
for approximately 1.6 million products. This keeps local inference and storage
manageable while providing a semantic representation beyond lexical matching.

In [17]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Model:", MODEL_NAME)
print("Device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


In [21]:
print("Embedding dimension:", model.get_embedding_dimension())
print("Max sequence length:", model.max_seq_length)

Embedding dimension: 384
Max sequence length: 256


In [19]:
tokenizer = model.tokenizer

print("Tokenizer:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)

Tokenizer: BertTokenizer
Model max length: 256


In [20]:
sample_texts = sample_products["product_text"].tolist()

sample_embeddings = model.encode(
    sample_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Embedding shape:", sample_embeddings.shape)
print("Data type:", sample_embeddings.dtype)

Embedding shape: (5, 384)
Data type: float32


In [22]:
print(tokenizer.model_max_length)
print(tokenizer.special_tokens_map)
print(tokenizer.num_special_tokens_to_add(pair=False))

256
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
2


In [23]:
sample_products["token_count"] = sample_products["product_text"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

sample_products[["parent_asin", "token_count"]]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (259 > 256). Running this sequence through the model will result in indexing errors


,parent_asin,token_count
0,B00MCW7G9M,192
1,B00YT6XQSE,64
2,B07SM135LS,259
3,B089CNGZCW,335
4,B004E2Z88O,107


## Field-aware token truncation

The MiniLM encoder accepts a maximum of 256 tokens, including two special
tokens. Therefore, 254 tokens are available for product content.

Instead of truncating the combined product text blindly, we assign separate
maximum budgets to each metadata field:

- Title: 64 tokens
- Categories: 32 tokens
- Features: 90 tokens
- Description: 54 tokens

These are maximum allocations rather than requirements. Missing fields are
omitted, and shorter fields do not need to be padded with artificial content.

This protects product identity and category information from being displaced
by unusually long descriptions.

In [29]:
FIELD_TOKEN_BUDGETS = {
    "title": 64,
    "categories": 32,
    "features": 90,
    "description": 54,
}

In [30]:
def truncate_field(text, max_tokens):
    if not text:
        return ""

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens,
    )["input_ids"]

    return tokenizer.decode(
        token_ids,
        skip_special_tokens=True,
    )

In [31]:
def build_truncated_product_text(row):
    fields = {
        "title": clean_text(row["title"]),
        "categories": " > ".join(
            clean_text(category)
            for category in row["categories"]
            if clean_text(category)
        ),
        "features": clean_text(row["features"]),
        "description": clean_text(row["description"]),
    }

    sections = []

    for field_name in [
        "title",
        "categories",
        "features",
        "description",
    ]:
        text = truncate_field(
            fields[field_name],
            FIELD_TOKEN_BUDGETS[field_name]
        )

        if text:
            sections.append(
                f"{field_name.capitalize()}: {text}"
            )

    return "\n".join(sections)

In [32]:
sample_products["truncated_product_text"] = sample_products.apply(
    build_truncated_product_text,
    axis=1
)

for _, row in sample_products[
    ["parent_asin", "truncated_product_text"]
].iterrows():
    print("=" * 80)
    print(row["parent_asin"])
    print(row["truncated_product_text"])

B00MCW7G9M
Title: fs - 1051 fatshark teleporter v3 headset
Categories: electronics > television & video > video glasses
Description: teleporter v3 the “ teleporter v3 ” kit sets a new level of value in the fpv world with fat shark renowned performance and quality. the fun of fpv is experienced firsthand through the large screen fpv heads
B00YT6XQSE
Title: ce - h22b12 - s1 4kx2k hdmi 4port
Categories: electronics > television & video > accessories > cables > hdmi cables
Features: upc : 662774021904 weight : 0. 600 lbs
Description: hdmi in - hdmi out
B07SM135LS
Title: digi - tatoo decal skin compatible with macbook pro 13 inch ( model a2338 / a2289 / a2251 ) - protective and decorative full body laptop skin decal sticker, anti - scratch vinly skin sticker wrap [ fresh marble ]
Categories: electronics > computers & accessories > laptop accessories > skins & decals > decals
Features: warning : please identify model number on the bottom of your macbook. only fits for model a2338 / a2289 / a

In [33]:
sample_products["truncated_token_count"] = (
    sample_products["truncated_product_text"].apply(
        lambda text: len(
            tokenizer(
                text,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )
    )
)

sample_products[
    ["parent_asin", "token_count", "truncated_token_count"]
]

,parent_asin,token_count,truncated_token_count
0,B00MCW7G9M,192,86
1,B00YT6XQSE,64,64
2,B07SM135LS,259,170
3,B089CNGZCW,335,153
4,B004E2Z88O,107,107


In [34]:
sample_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    categories
FROM read_json_auto('{METADATA_PATH}')
LIMIT 1000
"""

sample_products = con.execute(sample_query).df()

In [35]:
sample_products["truncated_product_text"] = sample_products.apply(
    build_truncated_product_text,
    axis=1
)

In [36]:
sample_products["truncated_token_count"] = (
    sample_products["truncated_product_text"].apply(
        lambda text: len(
            tokenizer(
                text,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )
    )
)

sample_products["truncated_token_count"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    1000.000000
mean      137.031000
std        58.613066
min         7.000000
50%       146.000000
90%       211.100000
95%       220.050000
99%       232.010000
max       241.000000
Name: truncated_token_count, dtype: float64

In [37]:
print(
    "Maximum tokens:",
    sample_products["truncated_token_count"].max()
)

print(
    "Products over limit:",
    (
        sample_products["truncated_token_count"] > 256
    ).sum()
)

Maximum tokens: 241
Products over limit: 0


### Semantic content representation

- Chose `sentence-transformers/all-MiniLM-L6-v2` for semantic product embeddings.
- The model produces 384-dimensional float32 embeddings with a 256-token
  maximum sequence length.
- Implemented field-aware token truncation rather than blindly truncating the
  combined product text.
- Final field budgets are 64 title, 32 categories, 90 features, and 54
  description tokens.
- Validated the preprocessing pipeline on 1,000 products with zero products
  exceeding the model's token limit.

## Embedding throughput benchmark

Before generating embeddings for the full catalog, we benchmark the semantic
encoder on a 10,000-product sample.

The benchmark measures embedding throughput on the local GPU so that the
expected runtime for the full catalog can be estimated before committing to
the large batch job.

Different batch sizes are tested because GPU utilization and throughput can
change significantly with batch size.

In [38]:
benchmark_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    categories
FROM read_json_auto('{METADATA_PATH}')
LIMIT 10000
"""

benchmark_products = con.execute(benchmark_query).df()

benchmark_products["product_text"] = benchmark_products.apply(
    build_truncated_product_text,
    axis=1
)

benchmark_texts = benchmark_products["product_text"].tolist()

print("Products:", len(benchmark_texts))

Products: 10000


In [39]:
_ = model.encode(
    benchmark_texts[:32],
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

In [40]:
import time

batch_sizes = [16, 32, 64, 128]

benchmark_results = []

for batch_size in batch_sizes:
    start_time = time.perf_counter()

    embeddings = model.encode(
        benchmark_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    elapsed = time.perf_counter() - start_time

    throughput = len(benchmark_texts) / elapsed

    benchmark_results.append({
        "batch_size": batch_size,
        "elapsed_seconds": elapsed,
        "products_per_second": throughput,
    })

    print(
        f"batch_size={batch_size:>3} | "
        f"time={elapsed:>8.2f}s | "
        f"throughput={throughput:>8.2f} products/s"
    )

batch_size= 16 | time=   15.70s | throughput=  637.11 products/s
batch_size= 32 | time=   15.56s | throughput=  642.82 products/s
batch_size= 64 | time=   15.82s | throughput=  632.30 products/s
batch_size=128 | time=   16.20s | throughput=  617.15 products/s


In [41]:
best_result = max(
    benchmark_results,
    key=lambda x: x["products_per_second"]
)

catalog_size = 1_609_860

estimated_seconds = (
    catalog_size / best_result["products_per_second"]
)

estimated_hours = estimated_seconds / 3600

print("Best batch size:", best_result["batch_size"])
print(
    "Estimated full-catalog time:",
    f"{estimated_hours:.2f} hours"
)

Best batch size: 32
Estimated full-catalog time: 0.70 hours


In [42]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent
metadata_path = PROJECT_ROOT / "data" / "raw" / "meta_Electronics.jsonl.gz"

metadata_duplicate_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_metadata_rows,
        COUNT(DISTINCT parent_asin) AS unique_parent_asins,
        COUNT(*) - COUNT(DISTINCT parent_asin) AS duplicate_rows
    FROM read_json_auto('{metadata_path}')
""").df()

metadata_duplicate_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_metadata_rows,unique_parent_asins,duplicate_rows
0,1610012,1610012,0


In [43]:
catalog = con.sql(f"""
    SELECT DISTINCT parent_asin
    FROM read_parquet('{PROJECT_ROOT / "data" / "processed" / "electronics_interactions" / "*.parquet"}')
    ORDER BY parent_asin
""").df()

catalog["item_idx"] = np.arange(len(catalog), dtype=np.int32)

catalog = catalog[["item_idx", "parent_asin"]]

catalog.head(), catalog.shape

(   item_idx parent_asin
 0         0  0007182147
 1         1  0008158851
 2         2  0008537909
 3         3  0011300000
 4         4  0060009810,
 (1609860, 2))

In [44]:
catalog_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "product_catalog.parquet"
)

catalog.to_parquet(catalog_path, index=False)

catalog.head()

,item_idx,parent_asin
0,0,0007182147
1,1,0008158851
2,2,0008537909
3,3,0011300000
4,4,0060009810


In [45]:
assert catalog["item_idx"].is_unique
assert catalog["parent_asin"].is_unique
assert catalog["item_idx"].iloc[0] == 0
assert catalog["item_idx"].iloc[-1] == len(catalog) - 1

print(f"Catalog size: {len(catalog):,}")
print(f"Index range: {catalog.item_idx.min()} → {catalog.item_idx.max()}")
print("Catalog mapping validation passed.")

Catalog size: 1,609,860
Index range: 0 → 1609859
Catalog mapping validation passed.
